In [0]:
#| default_exp edit_interactive

## Edit-interactive plan execution

Run a tiny Lisette agent loop against one notebook at a time. The inner agent receives the user plan plus a single compact notebook view, then can mutate only that notebook through scoped tools.

In [ ]:
#| export
import difflib
import os
from contextlib import redirect_stderr, redirect_stdout
from dataclasses import dataclass, field
from io import StringIO
from pathlib import Path

from fastcore.nbio import mk_cell
from fastcore.nbio import read_nb as _read_nb
from fastcore.nbio import write_nb as _write_nb
from nbdev.doclinks import nbdev_export

from nbskill.execute import exec_nb as _exec_nb
from nbskill.foundation import (
    _cell_hash, _cell_matches_hash, _cell_source, _clear_outputs,
    _parse_one_cell, _validate_code_cells,
)
from nbskill.parallel import notebook_locks
from nbskill.review import diff_nb as _diff_nb

In [0]:
#| export
EDIT_INTERACTIVE_SYSTEM = """\
You are nbskill edit-interactive. You edit exactly one notebook.

You receive a plan and one notebook view. Use only the provided notebook tools.
Prefer id-based edits with source_hash when the view gives one. Keep changes
small and aligned with the plan. Run cells when that is needed to verify the
work. Stop when the plan is complete and summarize what changed.
"""

In [0]:
#| export
def capture_call_text(func, **kwargs):
    "Run `func` and return captured stdout/stderr, or the return value."
    out, err = StringIO(), StringIO()
    with redirect_stdout(out), redirect_stderr(err):
        result = func(**kwargs)
    chunks = []
    if out.getvalue(): chunks.append(out.getvalue().rstrip())
    if err.getvalue(): chunks.append(err.getvalue().rstrip())
    if result is not None and not chunks: chunks.append(str(result))
    return "\n".join(chunk for chunk in chunks if chunk)

In [ ]:
#| export
def notebook_view(path, revision=0):
    "Render one notebook as a compact, stable text view."
    path = Path(path)
    with notebook_locks(path):
        nb = _read_nb(path)
        lines = [f"Notebook: {path}", f"Revision: {revision}", ""]
        for idx, cell in enumerate(nb.cells):
            lines.append(f"CELL {idx} id={cell.id} type={cell.cell_type} hash={_cell_hash(cell)}")
            lines.append("<<<SOURCE")
            lines.append(_cell_source(cell).rstrip())
            lines.append("SOURCE")
            lines.append("")
        return "\n".join(lines).rstrip() + "\n"

In [ ]:
#| export
@dataclass
class EditSession:
    "Mutable state for one edit-interactive notebook run."
    path: Path
    timeout: int = 30
    export: bool = True
    revision: int = 0
    log: list[str] = field(default_factory=list)
    tool_log: list[str] = field(default_factory=list)
    chat: object | None = None
    notebook_msg_idx: int | None = None

    def refresh_view(self):
        "Replace the notebook-view message in the Lisette history."
        if self.chat is None or self.notebook_msg_idx is None: return
        msg = self.chat.hist[self.notebook_msg_idx]
        view = notebook_view(self.path, self.revision)
        if isinstance(msg, dict): msg["content"] = view
        else: msg.content = view

    def record(self, message):
        "Append an operation to the session log."
        self.log.append(f"r{self.revision}: {message}")

    def record_tool(self, name, detail=""):
        "Append one tool use to the session log."
        suffix = f"({detail})" if detail else "()"
        self.tool_log.append(f"r{self.revision}: {name}{suffix}")

In [ ]:
#| export
def _save_notebook(nb, path, export=True):
    with notebook_locks(path):
        _write_nb(nb, path)
        if export: nbdev_export(path=str(path))

In [0]:
#| export
def _none_if_blank(value):
    if value is None: return None
    value = str(value)
    return None if value.strip().lower() in {"", "none", "null"} else value

In [0]:
#| export
def _edit_find_cell_by_id(cells, cell_id):
    matches = [(idx, cell) for idx, cell in enumerate(cells) if getattr(cell, "id", None) == str(cell_id)]
    if len(matches) == 1: return matches[0]
    if not matches: raise ValueError(f"No cell has id {cell_id!r}")
    raise ValueError(f"Multiple cells have id {cell_id!r}")

In [0]:
#| export
def _source_diff(before, after, label):
    diff = difflib.unified_diff(
        before.splitlines(True), after.splitlines(True),
        fromfile=f"{label}:before", tofile=f"{label}:after",
    )
    text = "".join(diff).strip()
    return text or "No source changes"

In [0]:
#| export
def _finish_write(session, nb, message, diff):
    _save_notebook(nb, session.path, export=session.export)
    session.revision += 1
    session.record(message)
    session.refresh_view()
    return f"{message}\nrevision={session.revision}\n\n{diff}"

In [0]:
#| export
def _validate_cell(cell):
    _validate_code_cells([cell])
    return cell

In [ ]:
#| export
def make_edit_tools(session):
    "Create notebook-scoped tools for one edit-interactive session."

    def add_cell(
        after_id: str | None = None,  # Cell id to insert after; blank/None appends
        content: str = "",  # New cell source; may start with %%markdown, %%code, or %%raw
    ) -> str:
        "Add one cell to the current notebook."
        with notebook_locks(session.path):
            nb = _read_nb(session.path)
            new_cell = _validate_cell(_parse_one_cell(content, "code"))
            anchor = _none_if_blank(after_id)
            target = len(nb.cells)
            if anchor is not None:
                idx, _ = _edit_find_cell_by_id(nb.cells, anchor)
                target = idx + 1
            session.record_tool("add_cell", f"after_id={anchor!r}")
            nb.cells.insert(target, new_cell)
            src = _cell_source(new_cell)
            where = f"after id={anchor}" if anchor is not None else "at end"
            msg = f"Added cell id={new_cell.id} {where}"
            return _finish_write(session, nb, msg, _source_diff("", src, f"cell {new_cell.id}"))

    def edit_cell(
        id: str,  # Cell id to edit
        new_src: str,  # Replacement source, or substring replacement when old_src is set
        old_src: str | None = None,  # Optional exact text to replace inside the cell
        source_hash: str | None = None,  # Optional expected source hash prefix
    ) -> str:
        "Edit one cell in the current notebook."
        with notebook_locks(session.path):
            nb = _read_nb(session.path)
            idx, cell = _edit_find_cell_by_id(nb.cells, id)
            expected_hash = _none_if_blank(source_hash)
            if not _cell_matches_hash(cell, expected_hash):
                actual = _cell_hash(cell, n=None)
                raise ValueError(f"Hash mismatch for id={id}: expected {expected_hash}, actual {actual[:12]}")
            before = _cell_source(cell)
            old_src = _none_if_blank(old_src)
            session.record_tool(
                "edit_cell",
                f"id={id!r}, old_src={'yes' if old_src is not None else 'no'}, source_hash={expected_hash!r}",
            )
            if old_src is None:
                new_cell = _validate_cell(_parse_one_cell(new_src, cell.cell_type))
                _clear_outputs(new_cell)
                new_cell.id = cell.id
                nb.cells[idx] = new_cell
                after = _cell_source(new_cell)
                mode = "cell"
            else:
                if old_src not in before: raise ValueError(f"old_src was not found in id={id}")
                after = before.replace(old_src, new_src, 1)
                if cell.cell_type == "code": _validate_cell(mk_cell(after, cell_type="code"))
                cell.source = after
                _clear_outputs(cell)
                mode = "text"
            msg = f"Edited {mode} id={id}"
            return _finish_write(session, nb, msg, _source_diff(before, after, f"cell {id}"))

    def run_cell(
        id: str,  # Cell id to execute through
    ) -> str:
        "Run notebook cells up to and including this cell id."
        with notebook_locks(session.path):
            _edit_find_cell_by_id(_read_nb(session.path).cells, id)
            session.record_tool("run_cell", f"id={id!r}")
            text = capture_call_text(
                _exec_nb, path=str(session.path), dest=str(session.path), up2id=str(id),
                timeout=session.timeout, show_output=True, verbose=False,
            )
            status = "error" if "Traceback (most recent call last)" in text else "ok"
            session.record(f"Ran cells through id={id} ({status})")
            return f"status={status}\n{text}" if text else f"Executed through id={id} ({status})"

    def remove_cell(
        id: str,  # Cell id to remove
        source_hash: str | None = None,  # Optional expected source hash prefix
    ) -> str:
        "Remove one cell from the current notebook."
        with notebook_locks(session.path):
            nb = _read_nb(session.path)
            idx, cell = _edit_find_cell_by_id(nb.cells, id)
            expected_hash = _none_if_blank(source_hash)
            if not _cell_matches_hash(cell, expected_hash):
                actual = _cell_hash(cell, n=None)
                raise ValueError(f"Hash mismatch for id={id}: expected {expected_hash}, actual {actual[:12]}")
            session.record_tool("remove_cell", f"id={id!r}, source_hash={expected_hash!r}")
            before = _cell_source(cell)
            del nb.cells[idx]
            msg = f"Removed cell id={id}"
            return _finish_write(session, nb, msg, _source_diff(before, "", f"cell {id}"))

    return [add_cell, edit_cell, run_cell, remove_cell]

In [ ]:
#| export
def make_chat(model, tools, hist, system_prompt=EDIT_INTERACTIVE_SYSTEM):
    "Create the Lisette chat object for edit-interactive."
    from lisette import Chat
    return Chat(model, sp=system_prompt, tools=tools, hist=hist, stream=str(model).startswith("chatgpt/"))

In [0]:
#| export
def response_text(response):
    "Extract readable text from a Lisette response or response list."
    if isinstance(response, list) and response: response = response[-1]
    try:
        message = response.choices[0].message
        content = message.content
    except (AttributeError, IndexError, TypeError):
        return "" if response is None else str(response)
    if isinstance(content, list):
        return "".join(str(item.get("text", item)) if isinstance(item, dict) else str(item) for item in content)
    return "" if content is None else str(content)

In [ ]:
#| export
def final_diff(path):
    "Return a nbdev code-cell diff, or a clear unavailable message."
    try:
        with notebook_locks(path):
            return capture_call_text(_diff_nb, path=str(path))
    except BaseException as exc:
        detail = str(exc)
        if "Could not find notebook" in detail or "No git repository found" in detail:
            return (
                "Code-cell diff against HEAD is unavailable because this notebook has no git baseline. "
                "This is expected for new or untracked notebooks."
            )
        return f"Code-cell diff unavailable: {type(exc).__name__}: {exc}"

In [ ]:
#| export
def execute_plan(
    notebook: str,  # Path to the one notebook the inner agent may edit
    plan: str,  # Plan for the inner agent to execute
    model: str | None = None,  # Lisette/LiteLLM model; defaults via env then openai/gpt-4.1
    max_steps: int = 20,  # Maximum Lisette tool-loop steps
    timeout: int = 30,  # Per-cell execution timeout for run_cell
    export: bool = True,  # Run nbdev export after write tools
) -> str:
    "Execute `plan` against one notebook using a Lisette edit-interactive loop."
    path = Path(notebook)
    if not path.exists(): raise ValueError(f"Notebook does not exist: {notebook}")
    model = model or os.environ.get("NBSKILL_EDIT_MODEL") or "openai/gpt-4.1"
    session = EditSession(path=path, timeout=timeout, export=export)
    hist = [
        {"role": "user", "content": f"Plan:\n{plan}"},
        {"role": "user", "content": notebook_view(path, session.revision)},
    ]
    tools = make_edit_tools(session)
    chat = make_chat(model, tools=tools, hist=hist)
    session.chat = chat
    session.notebook_msg_idx = len(chat.hist) - 1
    session.refresh_view()
    result = chat(
        "Execute the plan using only the notebook tools. Stop when the plan is complete.",
        max_steps=max_steps, return_all=True,
    )
    if not isinstance(result, (list, str, bytes, dict)) and hasattr(result, "__next__"):
        result = list(result)
    sections = [
        "edit-interactive complete",
        "",
        "Final response:",
        response_text(result).strip() or "(no final response)",
        "",
        "Tools used:",
        "\n".join(session.tool_log) if session.tool_log else "(no tool calls)",
        "",
        "Operation log:",
        "\n".join(session.log) if session.log else "(no notebook operations)",
        "",
        f"Final revision: {session.revision}",
        "",
        "Notebook diff:",
        final_diff(path).strip(),
    ]
    return "\n".join(sections).rstrip()

In [ ]:
import tempfile
from pathlib import Path as _Path

import nbskill.edit_interactive as ei
from fastcore.nbio import mk_cell as _mk_cell
from fastcore.nbio import new_nb as _new_nb
from fastcore.nbio import write_nb as _write_tmp_nb


with tempfile.TemporaryDirectory() as td:
    path = _Path(td) / "sample.ipynb"
    _write_tmp_nb(_new_nb([_mk_cell("#| default_exp sample"), _mk_cell("x = 1")]), path)
    view = ei.notebook_view(path, revision=3)
    assert "Revision: 3" in view
    assert "type=code" in view
    assert "x = 1" in view

In [ ]:
import tempfile
from pathlib import Path as _Path

import nbskill.edit_interactive as ei
from fastcore.nbio import mk_cell as _mk_cell
from fastcore.nbio import new_nb as _new_nb
from fastcore.nbio import read_nb as _read_tmp_nb
from fastcore.nbio import write_nb as _write_tmp_nb


with tempfile.TemporaryDirectory() as td:
    path = _Path(td) / "sample.ipynb"
    first = _mk_cell("#| default_exp sample")
    second = _mk_cell("x = 1")
    _write_tmp_nb(_new_nb([first, second]), path)
    session = ei.EditSession(path=path, export=False)
    class FakeChat:
        def __init__(self): self.hist = [{"role": "user", "content": "plan"}, {"role": "user", "content": ei.notebook_view(path)}]
    session.chat = FakeChat()
    session.notebook_msg_idx = 1
    add_cell, edit_cell, run_cell, remove_cell = ei.make_edit_tools(session)
    added = add_cell(second.id, "y = 2")
    nb = _read_tmp_nb(path)
    assert "Added cell" in added
    assert len(nb.cells) == 3
    assert "y = 2" in session.chat.hist[1]["content"]
    new_id = nb.cells[-1].id
    edited = edit_cell(new_id, "y = 3", source_hash=ei._cell_hash(nb.cells[-1]))
    assert "Edited cell" in edited
    nb = _read_tmp_nb(path)
    assert nb.cells[-1].source == "y = 3"
    removed = remove_cell(new_id, source_hash=ei._cell_hash(nb.cells[-1]))
    assert "Removed cell" in removed
    assert len(_read_tmp_nb(path).cells) == 2
    assert session.revision == 3

In [ ]:
import tempfile
from pathlib import Path as _Path

import nbskill.edit_interactive as ei
from fastcore.nbio import mk_cell as _mk_cell
from fastcore.nbio import new_nb as _new_nb
from fastcore.nbio import write_nb as _write_tmp_nb


with tempfile.TemporaryDirectory() as td:
    path = _Path(td) / "sample.ipynb"
    cell = _mk_cell("x = 1")
    _write_tmp_nb(_new_nb([cell]), path)
    session = ei.EditSession(path=path, export=False)
    edit_cell = ei.make_edit_tools(session)[1]
    try:
        edit_cell(cell.id, "x = 2", source_hash="bad")
    except ValueError as exc:
        assert "Hash mismatch" in str(exc)
    else:
        raise AssertionError("expected hash mismatch")

In [ ]:
import tempfile
from pathlib import Path as _Path

import nbskill.edit_interactive as ei
from fastcore.nbio import mk_cell as _mk_cell
from fastcore.nbio import new_nb as _new_nb
from fastcore.nbio import write_nb as _write_tmp_nb


with tempfile.TemporaryDirectory() as td:
    path = _Path(td) / "sample.ipynb"
    _write_tmp_nb(_new_nb([_mk_cell("x = 1")]), path)
    session = ei.EditSession(path=path, export=False)
    remove_cell = ei.make_edit_tools(session)[3]
    try:
        remove_cell("missing")
    except ValueError as exc:
        assert "No cell has id" in str(exc)
    else:
        raise AssertionError("expected missing cell failure")

In [ ]:
import tempfile
from pathlib import Path as _Path

import nbskill.edit_interactive as ei
from fastcore.nbio import mk_cell as _mk_cell
from fastcore.nbio import new_nb as _new_nb
from fastcore.nbio import write_nb as _write_tmp_nb


with tempfile.TemporaryDirectory() as td:
    path = _Path(td) / "sample.ipynb"
    c1 = _mk_cell("x = 1")
    c2 = _mk_cell("print(x + 1)")
    _write_tmp_nb(_new_nb([c1, c2]), path)
    session = ei.EditSession(path=path, export=False, timeout=5)
    run_cell = ei.make_edit_tools(session)[2]
    text = run_cell(c2.id)
    assert "Executed" in text
    assert "2" in text

In [ ]:
import tempfile
from pathlib import Path as _Path

import nbskill.edit_interactive as ei
from fastcore.nbio import mk_cell as _mk_cell
from fastcore.nbio import new_nb as _new_nb
from fastcore.nbio import read_nb as _read_tmp_nb
from fastcore.nbio import write_nb as _write_tmp_nb


class FakeChat:
    last = None

    def __init__(self, model, sp, tools, hist):
        self.model, self.sp, self.tools, self.hist = model, sp, tools, hist
        FakeChat.last = self

    def __call__(self, msg, max_steps=20, return_all=False):
        add_cell = self.tools[0]
        add_cell(None, "answer = 42")
        return "done"


old_make_chat = ei.make_chat
try:
    ei.make_chat = lambda model, tools, hist, system_prompt=ei.EDIT_INTERACTIVE_SYSTEM: FakeChat(model, system_prompt, tools, hist)
    with tempfile.TemporaryDirectory() as td:
        path = _Path(td) / "sample.ipynb"
        _write_tmp_nb(_new_nb([_mk_cell("#| default_exp sample")]), path)
        text = ei.execute_plan(str(path), "Add an answer cell.", model="fake", export=False)
        assert "Final response:" in text
        assert "Tools used:" in text
        assert "add_cell" in text
        assert "done" in text
        assert "answer = 42" in _read_tmp_nb(path).cells[-1].source
        assert FakeChat.last.hist[1]["content"].count("answer = 42") == 1
finally:
    ei.make_chat = old_make_chat

In [2]:
import json,httpx
from pathlib import Path
auth = json.loads(Path('~/.codex/auth.json').expanduser().read_text())
tok = auth['tokens']['access_token']
r = httpx.get("https://chatgpt.com/backend-api/codex/models?client_version=1.0.0",
              headers={"Authorization": f"Bearer {tok}"}, timeout=10).json()
' '.join(r['models'][0])

'prefer_websockets support_verbosity default_verbosity apply_patch_tool_type web_search_tool_type input_modalities supports_image_detail_original truncation_policy supports_parallel_tool_calls context_window max_context_window auto_compact_token_limit reasoning_summary_format default_reasoning_summary slug display_name description default_reasoning_level supported_reasoning_levels shell_type visibility minimal_client_version supported_in_api availability_nux upgrade priority base_instructions model_messages experimental_supported_tools available_in_plans supports_search_tool service_tiers additional_speed_tiers supports_reasoning_summaries'